# ¿Dónde se pierde rentabilidad? Análisis de Superstore Sales

**Pregunta de negocio:** ¿qué categorías, productos y niveles de descuento destruyen margen, y qué debería cambiar la gerencia comercial?

**Datos:** Superstore Sales (Kaggle). **Stack:** Python (pandas, matplotlib).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = "{:,.2f}".format
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})

RAW = Path("../data/raw/Sample - Superstore.csv")   # ajusta el nombre si tu archivo es distinto
FIG = Path("../reports/figures")
OUT = Path("../reports")

## 1. Carga y revisión de calidad de datos

In [ ]:
df = pd.read_csv(RAW, encoding="latin-1")          # el archivo original viene en latin-1
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))

esperadas = {"order_id", "order_date", "ship_date", "segment", "region", "category",
             "sub_category", "product_name", "sales", "quantity", "discount", "profit"}
faltan = esperadas - set(df.columns)
print("Filas:", len(df), "| Columnas:", len(df.columns))
print("Columnas faltantes:", faltan or "ninguna")
df.head()

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["ship_date"] = pd.to_datetime(df["ship_date"], errors="coerce")

print("Duplicados exactos:", df.duplicated().sum())
print("Fechas inválidas:", df["order_date"].isna().sum())
print(df.isna().sum()[lambda s: s > 0])
df = df.drop_duplicates()

df["margen"] = df["profit"] / df["sales"]
df["dias_envio"] = (df["ship_date"] - df["order_date"]).dt.days

## 2. KPI generales

In [ ]:
ventas, utilidad = df["sales"].sum(), df["profit"].sum()
kpis = pd.Series({
    "Ventas": ventas,
    "Utilidad": utilidad,
    "Margen %": utilidad / ventas * 100,
    "Pedidos": df["order_id"].nunique(),
    "Ticket promedio": ventas / df["order_id"].nunique(),
    "% líneas con pérdida": (df["profit"] < 0).mean() * 100,
})
kpis

## 3. Rentabilidad por categoría y sub-categoría

In [ ]:
def resumen(g):
    t = df.groupby(g).agg(ventas=("sales", "sum"), utilidad=("profit", "sum"),
                          descuento_prom=("discount", "mean")).reset_index()
    t["margen_pct"] = t["utilidad"] / t["ventas"] * 100
    return t.sort_values("utilidad")

por_cat = resumen("category")
por_sub = resumen(["category", "sub_category"])
por_cat

In [ ]:
fig, ax = plt.subplots()
colores = ["#c0392b" if v < 0 else "#2e86ab" for v in por_sub["utilidad"]]
ax.barh(por_sub["sub_category"], por_sub["utilidad"], color=colores)
ax.set_title("Utilidad por sub-categoría (rojo = pérdida)")
ax.set_xlabel("Utilidad")
plt.tight_layout(); plt.savefig(FIG / "utilidad_subcategoria.png", dpi=150); plt.show()

## 4. ¿Los descuentos destruyen margen?

In [ ]:
bins = [-0.001, 0, 0.1, 0.2, 0.3, 0.5, 1]
labels = ["0%", "0-10%", "10-20%", "20-30%", "30-50%", ">50%"]
df["tramo_desc"] = pd.cut(df["discount"], bins=bins, labels=labels)

desc = (df.groupby("tramo_desc", observed=True)
          .agg(ventas=("sales", "sum"), utilidad=("profit", "sum"), lineas=("sales", "size"))
          .assign(margen_pct=lambda t: t["utilidad"] / t["ventas"] * 100)
          .reset_index())
desc

In [ ]:
fig, ax = plt.subplots()
ax.bar(desc["tramo_desc"].astype(str), desc["margen_pct"],
       color=["#c0392b" if v < 0 else "#2e86ab" for v in desc["margen_pct"]])
ax.axhline(0, color="black", lw=0.8)
ax.set_title("Margen % según tramo de descuento")
ax.set_ylabel("Margen %")
plt.tight_layout(); plt.savefig(FIG / "margen_vs_descuento.png", dpi=150); plt.show()

## 5. Productos y regiones con mayor pérdida

In [ ]:
peores = (df.groupby("product_name")["profit"].sum().sort_values().head(10).reset_index())
region = resumen("region")
display(peores); display(region)

## 6. Tendencia mensual

In [ ]:
mensual = (df.set_index("order_date").resample("MS")
             .agg(ventas=("sales", "sum"), utilidad=("profit", "sum")))
mensual["margen_pct"] = mensual["utilidad"] / mensual["ventas"] * 100

fig, ax = plt.subplots()
ax.plot(mensual.index, mensual["ventas"], label="Ventas")
ax.plot(mensual.index, mensual["utilidad"], label="Utilidad")
ax.set_title("Ventas y utilidad mensual"); ax.legend()
plt.tight_layout(); plt.savefig(FIG / "tendencia_mensual.png", dpi=150); plt.show()

## 7. Exportar tablas para Power BI

In [ ]:
df.to_csv(OUT / "ventas_limpio.csv", index=False)
por_sub.to_csv(OUT / "rentabilidad_subcategoria.csv", index=False)
desc.to_csv(OUT / "margen_por_descuento.csv", index=False)
mensual.reset_index().to_csv(OUT / "tendencia_mensual.csv", index=False)
print("Tablas exportadas en", OUT.resolve())

## 8. Conclusiones y recomendaciones

*Completa esto con TUS resultados; es la parte que más valora un reclutador.*

1. **Hallazgo 1:** (ej. la sub-categoría X pierde $__ pese a vender $__).
2. **Hallazgo 2:** (ej. con descuentos sobre __% el margen pasa a negativo).
3. **Hallazgo 3:** (ej. la región __ concentra __% de las pérdidas).

**Recomendaciones:** (ej. tope de descuento de __%, revisar precios de X, renegociar costo de Y).

**Impacto estimado:** si se eliminaran las líneas con pérdida por descuento, la utilidad subiría en $__ (__%).